<a href="https://colab.research.google.com/github/rymadinari/-arene-des-algos-Ryma-Dinari-/blob/main/arene.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
# Phase 1 : Charger et explorer le dataset
from sklearn.datasets import load_breast_cancer
import numpy as np

def explorer_dataset():

    # Charger le dataset
    X, y = load_breast_cancer(return_X_y=True)
    dataset = load_breast_cancer()

    # Afficher la forme de X
    lignes, colonnes = X.shape
    print(f"Lignes, colonnes : {X.shape}")

    # Compter et afficher les exemples par classe
    for i, nom_classe in enumerate(dataset.target_names):
        nb = np.sum(y == i)
        pct = nb / len(y) * 100
        print(f"Classe {i} ({nom_classe}) : {nb} cas ({pct:.1f}%)")


    #Checkpoint 1 — cas normal
    counts = [np.sum(y == i) for i in range(len(dataset.target_names))]
    max_pct = max(counts) / len(y) * 100
    if max_pct > 70:
        print(f"Dataset déséquilibré ! La classe majoritaire représente {max_pct:.1f}%")
    else:
        print("Dataset relativement équilibré")

    return X, y

X, y = explorer_dataset()

Lignes, colonnes : (569, 30)
Classe 0 (malignant) : 212 cas (37.3%)
Classe 1 (benign) : 357 cas (62.7%)
Dataset relativement équilibré


In [14]:
# Checkpoint 2 — cas limite : une seule classe
print("Checkpoint : une seule classe ")
X_filtre = X[y == 0]
y_filtre = y[y == 0]
print(f"Forme après filtre : {X_filtre.shape}")
print(f"Classes présentes : {np.unique(y_filtre)}")

# Checkpoint 3 — cas adversarial : dataset déséquilibré artificiel
print("Checkpoint : dataset déséquilibré (95/5)")
y_fake = np.array([0] * 950 + [1] * 50)
for i in range(2):
    nb = np.sum(y_fake == i)
    pct = nb / len(y_fake) * 100
    print(f"Classe {i} : {nb} cas ({pct:.1f}%)")
max_pct = max([np.sum(y_fake == i) for i in range(2)]) / len(y_fake) * 100
if max_pct > 70:
    print(f"Déséquilibré ! Classe majoritaire : {max_pct:.1f}%")

Checkpoint : une seule classe 
Forme après filtre : (212, 30)
Classes présentes : [0]
Checkpoint : dataset déséquilibré (95/5)
Classe 0 : 950 cas (95.0%)
Classe 1 : 50 cas (5.0%)
Déséquilibré ! Classe majoritaire : 95.0%


In [16]:
# Phase 2 : Pipeline supervisé complet
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier

def entrainer_et_evaluer(modele, X_train, X_test, y_train, y_test):

    modele.fit(X_train, y_train)

    y_pred = modele.predict(X_test)

    return accuracy_score(y_test, y_pred)



# Charger les données de la phase 1
X, y = explorer_dataset()

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print(f"Train : {X_train.shape[0]} exemples")
print(f"Test  : {X_test.shape[0]} exemples")

# Tester avec un arbre de décision
modele = DecisionTreeClassifier(random_state=42)
acc = entrainer_et_evaluer(modele, X_train, X_test, y_train, y_test)
print(f"\nAccuracy arbre de décision : {acc*100:.1f}%")

Lignes, colonnes : (569, 30)
Classe 0 (malignant) : 212 cas (37.3%)
Classe 1 (benign) : 357 cas (62.7%)
Dataset relativement équilibré
Train : 455 exemples
Test  : 114 exemples

Accuracy arbre de décision : 94.7%


In [21]:
# Phase 3 : L'Arène — classement de plusieurs algos
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

def arene(X_train, X_test, y_train, y_test):

    modeles = {
        "Régression logistique" : LogisticRegression(max_iter=10000, random_state=42),
        "KNN"                   : KNeighborsClassifier(),
        "Arbre de décision"     : DecisionTreeClassifier(random_state=42),
    }

    resultats = {}
    for nom, modele in modeles.items():
        acc = entrainer_et_evaluer(modele, X_train, X_test, y_train, y_test)
        resultats[nom] = acc

    # trier par accuracy décroissante et afficher le podium
    classement = sorted(resultats.items(), key=lambda x: x[1], reverse=True)

    for rang, (nom, acc) in enumerate(classement, start=1):
        print(f"{rang:<6} {nom:<25} {acc*100:>7.1f}%")

    return classement


#  Lancer l'Arène
classement = arene(X_train, X_test, y_train, y_test)

1      Régression logistique        95.6%
2      KNN                          95.6%
3      Arbre de décision            94.7%


In [23]:
# Phase 4 : Clustering non supervisé (KMeans)
from sklearn.cluster import KMeans
import numpy as np

def clustering_aveugle(X):
    kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
    return kmeans.fit_predict(X)


# Lancer le clustering
clusters = clustering_aveugle(X)

# Comparer avec les vraies étiquettes
from sklearn.metrics import adjusted_rand_score

score = adjusted_rand_score(y, clusters)
print(f"Score de similarité clusters vs vraies classes : {score:.3f}")

print("Correspondance clusters → vraies classes :")
for cluster in [0, 1]:
    masque = clusters == cluster
    nb_malin  = np.sum(y[masque] == 0)
    nb_benin  = np.sum(y[masque] == 1)
    total     = np.sum(masque)
    print(f"Cluster {cluster} ({total} points) : {nb_malin} malignes · {nb_benin} bénignes")

Score de similarité clusters vs vraies classes : 0.491
Correspondance clusters → vraies classes :
Cluster 0 (131 points) : 130 malignes · 1 bénignes
Cluster 1 (438 points) : 82 malignes · 356 bénignes


In [28]:
# Phase 5 : Changer de terrain — dataset Wine (3 classes)
from sklearn.datasets import load_wine

# Charger nouveau dataset
X_wine, y_wine = load_wine(return_X_y=True)
dataset_wine = load_wine()

print(f"Lignes, colonnes : {X_wine.shape}")
print()
for i, nom in enumerate(dataset_wine.target_names):
    nb  = np.sum(y_wine == i)
    pct = nb / len(y_wine) * 100
    print(f"Classe {i} ({nom}) : {nb} cas ({pct:.1f}%)")

# Split
X_train_w, X_test_w, y_train_w, y_test_w = train_test_split(
    X_wine, y_wine, test_size=0.2, random_state=42
)

print("Arène sur Wine")
classement_wine = arene(X_train_w, X_test_w, y_train_w, y_test_w)

Lignes, colonnes : (178, 13)

Classe 0 (class_0) : 59 cas (33.1%)
Classe 1 (class_1) : 71 cas (39.9%)
Classe 2 (class_2) : 48 cas (27.0%)
Arène sur Wine
1      Régression logistique       100.0%
2      Arbre de décision            94.4%
3      KNN                          72.2%
